# 🎧 Spotify Streaming History Analysis

## Objetivo do projeto

Este projeto tem como objetivo transformar meu histórico pessoal de streaming do Spotify (2020–2026) em um dataset estruturado para Business Intelligence.

A proposta envolve:

- Extração de múltiplos arquivos JSON
- Sanitização de dados sensíveis
- Padronização e transformação com Python
- Criação de dataset analítico para Power BI

---

### Ferramentas utilizadas

- Python
- Pandas
- Jupyter Notebook
- Power BI

---

### Resultado final

Dashboard interativo focado em:

- Comportamento de consumo
- Evolução musical ao longo do tempo
- Retenção por artista
- Preferência de dispositivo

In [ ]:
import pandas as pd
from pathlib import Path

# 1. Entendimento da fonte de dados

Os dados foram exportados diretamente do Spotify através do recurso de solicitação de dados pessoais.

A estrutura original contém múltiplos arquivos JSON, separados por períodos anuais, abrangendo os anos de 2020 até 2026.

Cada registro representa uma reprodução realizada pelo usuário/por mim.

# 2. Sanitização de dados sensíveis

Por se tratar de dados pessoais, alguns campos foram removidos antes da análise.

Campos removidos:

- IP Address (`ip_addr`)

Objetivo:

Garantir privacidade e tornar o projeto seguro para publicação pública no GitHub.

In [3]:
# Definindo o caminho da pasta com os arquivos JSON
caminho_sanitized = Path("../data/sanitized")

# Buscando arquivos JSON
arquivos_json = list(caminho_sanitized.glob("*.json"))

print("Arquivos encontrados:", len(arquivos_json))
arquivos_json

Arquivos encontrados: 9


[WindowsPath('../data/sanitized/Streaming_History_Audio_2020.json'),
 WindowsPath('../data/sanitized/Streaming_History_Audio_2021.json'),
 WindowsPath('../data/sanitized/Streaming_History_Audio_2022.json'),
 WindowsPath('../data/sanitized/Streaming_History_Audio_2023.json'),
 WindowsPath('../data/sanitized/Streaming_History_Audio_2024.json'),
 WindowsPath('../data/sanitized/Streaming_History_Audio_2025.json'),
 WindowsPath('../data/sanitized/Streaming_History_Audio_2026.json'),
 WindowsPath('../data/sanitized/Streaming_History_Video_2025.json'),
 WindowsPath('../data/sanitized/Streaming_History_Video_2026.json')]

# 3. Consolidação dos arquivos JSON

Nesta etapa, todos os arquivos JSON são carregados e unificados em um único DataFrame.

Objetivo:

Criar uma base histórica consolidada para facilitar transformação e modelagem analítica.

In [4]:
# Lista para aqmazenar os DataFrames
dfs = []

# Lendos todos os JSONs
for arquivo in arquivos_json:
    df_temp = pd.read_json(arquivo)
    dfs.append(df_temp)

# Unificando todos os arquivos
df = pd.concat(dfs, ignore_index=True)

print(f"Total de registros: {df.shape[0]:,}")
print(f"Total de colunas: {df.shape[1]}")

Total de registros: 66,999
Total de colunas: 22


In [5]:
# Visualizando a estrutura
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 66999 entries, 0 to 66998
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   ts                                 66999 non-null  str    
 1   platform                           66999 non-null  str    
 2   ms_played                          66999 non-null  int64  
 3   conn_country                       66999 non-null  str    
 4   master_metadata_track_name         66830 non-null  str    
 5   master_metadata_album_artist_name  66830 non-null  str    
 6   master_metadata_album_album_name   66830 non-null  str    
 7   spotify_track_uri                  66830 non-null  str    
 8   episode_name                       169 non-null    object 
 9   episode_show_name                  169 non-null    object 
 10  spotify_episode_uri                169 non-null    object 
 11  audiobook_title                    0 non-null      float64
 12  a

# Biblioteca de Dados

- **ts:** Timestamp da reprodução. Data e hora exata em que o conteúdo foi tocado (formato UTC).
- **platform:** Plataforma/dispositivo utilizado para ouvir. Ex.: Android, Windows, Web Player, iPhone etc.
- **ms_played:** Tempo reproduzido em milissegundos. Ex.: 180000 = 3 minutos.
- **conn_country:** País da conexão no momento da reprodução. Ex.: BR = Brasil.
- **master_metadata_track_name:** Nome da faixa musical reproduzida.
- **master_metadata_album_artist_name:** Nome do artista principal da faixa.
- **master_metadata_album_album_name:** Nome do álbum da faixa.
- **spotify_track_uri:** Identificador único da música dentro do Spotify.
- **episode_name:**	Nome do episódio (quando o conteúdo for podcast).
- **episode_show_name:** Nome do programa/podcast.
- **spotify_episode_uri:** Identificador único do episódio no Spotify.
- **audiobook_title:** Nome do audiobook (se houver).
- **audiobook_uri:** Identificador do audiobook.
- **audiobook_chapter_uri:** Identificador do capítulo do audiobook.
- **audiobook_chapter_title:** Nome do capítulo do audiobook.
- **reason_start:** Motivo pelo qual a reprodução começou. Ex.: clique manual, autoplay, próxima faixa.
- **reason_end:** Motivo pelo qual a reprodução terminou. Ex.: música finalizada, pulada, logout.
- **shuffle:** Indica se a reprodução ocorreu no modo aleatório (True/False).
- **skipped:** Indica se a música foi pulada (True/False).
- **offline:** Indica se a reprodução ocorreu offline (True/False).
- **offline_timestamp:** Timestamp da sincronização offline (quando aplicável).
- **incognito_mode:** Indica se o modo privado/incógnito estava ativado.

In [6]:
# Criando uma cópia de trabalho
df_clean = df.copy()

# Mantendo apenas musicas
df_clean["master_metadata_track_name"].notna()

# Removendo colunas irrelevantes
colunas_remover = [
    "spotify_track_uri",
    "episode_name",
    "episode_show_name",
    "spotify_episode_uri",
    "audiobook_title",
    "audiobook_uri",
    "audiobook_chapter_uri",
    "audiobook_chapter_title",
    "offline_timestamp"
]

df_clean.drop(columns=colunas_remover, inplace=True)

print(f"Shape após limpeza: {df_clean.shape}")
df_clean.head()

Shape após limpeza: (66999, 13)


,ts,platform,ms_played,conn_country,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,reason_start,reason_end,shuffle,skipped,offline,incognito_mode
0,2020-10-12T20:26:07Z,"Android OS 9 API 28 (LGE, LM-K510)",60894,BR,Sem Saída,Sadstation,Sem Saída,trackdone,logout,True,False,False,False
1,2020-10-12T20:31:42Z,"Android OS 9 API 28 (LGE, LM-K510)",203501,BR,Thnks fr th Mmrs,Fall Out Boy,Infinity On High,clickrow,trackdone,True,False,False,False
2,2020-10-12T20:35:42Z,"Android OS 9 API 28 (LGE, LM-K510)",229925,BR,Feel Invincible,Skillet,Unleashed,trackdone,trackdone,True,False,False,False
3,2020-10-12T20:39:49Z,"Android OS 9 API 28 (LGE, LM-K510)",246760,BR,Let the Sparks Fly,Thousand Foot Krutch,The End Is Where We Begin,trackdone,trackdone,True,False,False,False
4,2020-10-12T20:43:40Z,"Android OS 9 API 28 (LGE, LM-K510)",230426,BR,Comatose,Skillet,Comatose,trackdone,trackdone,True,False,False,False


# 4. Engenharia de atributos

Nesta etapa, novas variáveis analíticas são criadas para enriquecer o dataset.

Principais features:

- Ano de reprodução
- Hora do dia
- Tipo de dispositivo
- Status de retenção (completa ou pulada)

Essas variáveis serão utilizadas posteriormente no dashboard.

In [7]:
# Convertendo datastamp para datetime
df_clean["ts"] = pd.to_datetime(df_clean["ts"])

# Criando colunas temporais
df_clean["data"] = df_clean["ts"].dt.date
df_clean["ano"] = df_clean["ts"].dt.year
df_clean["mes"] = df_clean["ts"].dt.month
df_clean["hora"] = df_clean["ts"].dt.hour

# Nomes dos meses e dias da semana
df_clean["nome_mes"] = df_clean["ts"].dt.month_name()
df_clean["dia_semana"] = df_clean["ts"].dt.day_name()

# Convertendo tempo ouvido
df_clean["minutos_played"] = df_clean["ms_played"] / 60000
df_clean["horas_played"] = df_clean["ms_played"] / 3600000

print("Colunas criadas com sucesso.")

Colunas criadas com sucesso.


# 5. Validação e qualidade dos dados

Após a transformação, são realizadas validações para garantir consistência da base:

- Distribuição por dispositivo
- Taxa de retenção
- Presença de valores nulos
- Integridade temporal

In [8]:
def categorizar_dispositivo(plataforma):
    
    plataforma = plataforma.lower()
    
    if "android" in plataforma or "iphone" in plataforma or "ios" in plataforma:
        return "Mobile"
    
    elif "windows" in plataforma or "mac" in plataforma or "linux" in plataforma:
        return "Desktop"
    
    elif "web" in plataforma:
        return "Web"
    
    else:
        return "Outros"


df_clean["tipo_dispositivo"] = df_clean["platform"].apply(categorizar_dispositivo)

df_clean["tipo_dispositivo"].value_counts()

tipo_dispositivo
Mobile     64578
Desktop     2420
Outros         1
Name: count, dtype: int64

In [9]:
# Criando flag de retenção
df_clean["retencao"] = df_clean["skipped"].apply(
    lambda x: "Completa" if x == False else "Pulada"
)

df_clean["retencao"].value_counts()

retencao
Completa    53415
Pulada      13584
Name: count, dtype: int64

In [10]:
df_clean["retencao_real"] = df_clean["ms_played"].apply(
    lambda x: "Pulada" if x < 30000 else "Retida"
)

df_clean["retencao_real"].value_counts()

retencao_real
Retida    52351
Pulada    14648
Name: count, dtype: int64

In [11]:
artistas_ano = (
    df_clean.groupby(
        ["ano", "master_metadata_album_artist_name"])
        ["minutos_played"].sum().reset_index().sort_values(
        ["ano", "minutos_played"],
        ascending=[True, False]
    )
)

artistas_ano.head(20)

,ano,master_metadata_album_artist_name,minutos_played
77,2020,Dri,1001.372583
160,2020,Kayou.,296.392017
306,2020,Thousand Foot Krutch,283.088550
277,2020,Skillet,272.943817
254,2020,Red,259.461450
307,2020,Three Days Grace,208.532900
78,2020,Dri.mp3,175.858700
50,2020,Chaoss,143.243667
191,2020,Magyn,140.206933
56,2020,Coldplay,129.873750


In [12]:
top_artistas_por_ano = (
    artistas_ano
    .groupby("ano")
    .head(1)
)

top_artistas_por_ano

,ano,master_metadata_album_artist_name,minutos_played
77,2020,Dri,1001.372583
708,2021,VMZ,2543.487267
1035,2022,NEFFEX,2722.357017
1341,2023,Djonga,1816.621033
2172,2024,Pearl Jam,1661.208967
2787,2025,MORADA,2252.204367
3562,2026,Sara Evelyn,1068.812817


In [ ]:
horarios = (
    df_clean.groupby("hora")["minutos_played"].sum().reset_index().sort_values("minutos_played",ascending=False)
)

horarios.head(10)

,hora,minutos_played
21,21,15365.494033
17,17,14452.523267
18,18,13277.203617
16,16,11695.745067
19,19,11622.113400
20,20,10445.634917
13,13,9918.580817
12,12,9602.735667
11,11,8776.841817
14,14,8723.458367


In [14]:
artistas_retencao = (
    df_clean
    .groupby("master_metadata_album_artist_name")
    .agg(
        minutos_total=("minutos_played", "sum"),
        total_reproducoes=("master_metadata_track_name", "count"),
        musicas_puladas=("skipped", "sum")
    )
    .reset_index()
)

# Taxa de retenção
artistas_retencao["taxa_retencao"] = (
    1 - (
        artistas_retencao["musicas_puladas"] /
        artistas_retencao["total_reproducoes"]
    )
) * 100

# Filtrar apenas artistas relevantes
artistas_retencao = artistas_retencao[
    artistas_retencao["total_reproducoes"] >= 100
]

# Ordenação mais inteligente
artistas_retencao = artistas_retencao.sort_values(
    ["taxa_retencao", "minutos_total"],
    ascending=[False, False]
)

artistas_retencao.head(10)

,master_metadata_album_artist_name,minutos_total,total_reproducoes,musicas_puladas,taxa_retencao
463,Dri.mp3,448.807933,236,0,100.000000
1500,Simon Curtis,339.763967,116,0,100.000000
56,After the Fall,319.324617,180,0,100.000000
295,Chaoss,423.165400,258,2,99.224806
1231,OSRSBeatz,200.333400,109,1,99.082569
66,Alan Walker,204.682533,106,1,99.056604
978,Lucas Hype,233.781900,103,1,99.029126
1719,Valentina Matheus,393.419383,169,2,98.816568
84,Ali Gatie,1183.174317,331,4,98.791541
865,Kayou.,383.003150,149,2,98.657718


# 6. Exportação para Business Intelligence

A base tratada é exportada em formato CSV para consumo no Power BI.

O dataset final servirá como camada analítica do dashboard.

In [ ]:
df_clean.to_csv(
    "../data/processed/spotify_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset exportado com sucesso.")

Dataset exportado com sucesso.


In [16]:
df_clean.head()

,ts,platform,ms_played,conn_country,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,reason_start,reason_end,shuffle,...,ano,mes,hora,nome_mes,dia_semana,minutos_played,horas_played,tipo_dispositivo,retencao,retencao_real
0,2020-10-12 20:26:07+00:00,"Android OS 9 API 28 (LGE, LM-K510)",60894,BR,Sem Saída,Sadstation,Sem Saída,trackdone,logout,True,...,2020,10,20,October,Monday,1.014900,0.016915,Mobile,Completa,Retida
1,2020-10-12 20:31:42+00:00,"Android OS 9 API 28 (LGE, LM-K510)",203501,BR,Thnks fr th Mmrs,Fall Out Boy,Infinity On High,clickrow,trackdone,True,...,2020,10,20,October,Monday,3.391683,0.056528,Mobile,Completa,Retida
2,2020-10-12 20:35:42+00:00,"Android OS 9 API 28 (LGE, LM-K510)",229925,BR,Feel Invincible,Skillet,Unleashed,trackdone,trackdone,True,...,2020,10,20,October,Monday,3.832083,0.063868,Mobile,Completa,Retida
3,2020-10-12 20:39:49+00:00,"Android OS 9 API 28 (LGE, LM-K510)",246760,BR,Let the Sparks Fly,Thousand Foot Krutch,The End Is Where We Begin,trackdone,trackdone,True,...,2020,10,20,October,Monday,4.112667,0.068544,Mobile,Completa,Retida
4,2020-10-12 20:43:40+00:00,"Android OS 9 API 28 (LGE, LM-K510)",230426,BR,Comatose,Skillet,Comatose,trackdone,trackdone,True,...,2020,10,20,October,Monday,3.840433,0.064007,Mobile,Completa,Retida
